In [1]:
import requests
from dotenv import load_dotenv
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_classic import hub

load_dotenv()

/home/dhruv-kapri/Desktop/Projects/langchain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
llm = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash-lite')

In [3]:
search_tool = DuckDuckGoSearchRun()

@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=4d1d8ae207a8c845a52df8a67bf3623e&query={city}'

  response = requests.get(url)

  return response.json()

In [4]:
# Step 2: Pull the ReAct prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")  # pulls the standard ReAct agent prompt

In [5]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm = llm,
    tools = [get_weather_data, search_tool],
    prompt = prompt,
)

In [6]:
# Step 4: Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent = agent,
    tools = [get_weather_data, search_tool],
    verbrose = True
)

In [7]:
# Step 5: Invoke
# response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})
response = agent_executor.invoke({"input": "Find the current weather condition of capital of Madhya Pradesh."})
print(response)

{'input': 'Find the current weather condition of capital of Madhya Pradesh.', 'output': 'The current weather condition in Bhopal, the capital of Madhya Pradesh, is clear with a temperature of 16 degrees.'}


In [8]:
response['output']

'The current weather condition in Bhopal, the capital of Madhya Pradesh, is clear with a temperature of 16 degrees.'